# SVEF Project: Exploratory Data Analysis (Check-in 1)
**Project:** Safety-Validated/Efficacy-Failed (SVEF) Drug Rescue Pipeline  
**Course:** BIFX546  
**Environment:** Google Colab

## 1. Setup and Data Ingestion
**Instructions for Manual Upload:**
1. Click the **Folder icon** on the left sidebar of Google Colab.
2. Drag and drop `SVEF_Enriched_Final.csv` and `svef_logic_audit.csv` into that pane.
3. Wait for the upload progress circles to finish.
4. Run the cell below to load the data.

In [ ]:
!pip install matplotlib-venn seaborn pandas numpy
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib_venn import venn2

# File paths in Colab root directory
enriched_path = 'SVEF_Enriched_Final.csv'
audit_path = 'svef_logic_audit.csv'

def check_and_load(path):
    if os.path.exists(path):
        print(f"Success: Found {path}")
        return pd.read_csv(path)
    else:
        print(f"Error: {path} NOT FOUND in the file pane. Please upload it manually.")
        return None

df = check_and_load(enriched_path)
audit = check_and_load(audit_path)

if df is not None and audit is not None:
    print("\nAll datasets loaded successfully!")

## 2. Dataset Overview
We will inspect the enriched dataset to see how the pipeline performed.

In [ ]:
if df is not None:
    print(f"Enriched Dataset Shape: {df.shape}")
    print(f"Unique Trials (NCT IDs): {df['nct_id'].nunique()}")
    display(df.head())

## 3. Summary Statistics
Analysis of recovery rates and safety exposure.

In [ ]:
if df is not None:
    stats = {
        "Total Candidate Rows": len(df),
        "Unique Small Molecules": df['name'].nunique(),
        "SMILES Recovery Rate": f"{(df['is_dti_ready'].sum() / len(df)) * 100:.2f}%",
        "Average Enrollment": round(df['enrollment'].mean(), 2),
        "Mean Safety Score": round(df['Safety_Score'].mean(), 4)
    }

    for key, val in stats.items():
        print(f"{key}: {val}")

## 4. Visualizations
### A. Multi-Status Distribution
Distribution of clinical trial statuses.

In [ ]:
if df is not None:
    plt.figure(figsize=(10, 6))
    sns.countplot(data=df, x='overall_status', palette='viridis', hue='overall_status', legend=False)
    plt.title("Distribution of Clinical Trial Statuses in SVEF Pool")
    plt.ylabel("Number of Trial-Drug Combinations")
    plt.show()

### B. Audit Status Outcomes
Categorization of trials into Efficacy vs. Strategic exits.

In [ ]:
if df is not None:
    plt.figure(figsize=(12, 6))
    audit_counts = df['audit_status'].value_counts()
    audit_counts.plot(kind='pie', autopct='%1.1f%%', colormap='Set3')
    plt.title("Categorization of SVEF Candidates")
    plt.ylabel("")
    plt.show()

### C. Safety Score Distribution
Safety Score histogram representing human exposure.

In [ ]:
if df is not None:
    plt.figure(figsize=(10, 6))
    sns.histplot(df['Safety_Score'].dropna(), bins=30, kde=True, color='blue')
    plt.title("Human Safety Exposure (Normalized Safety Score)")
    plt.xlabel("Safety Score (0.0 = Low, 1.0 = High)")
    plt.show()

## 5. Progress Note
**Current Status:**  
The SVEF pipeline has been successfully upgraded from a basic keyword filter to a **Unified Multi-Status Audit System**. We now process Terminated, Suspended, Withdrawn, and Unknown trials across Phases 2 and 3. 

**Key Achievements:**
1. **Auditability:** Every inclusion/exclusion is now logged with a specific keyword trigger (e.g., 'futility', 'not related to safety').
2. **Feature Enrichment:** Automated integration with PubChem PUG-REST API for SMILES retrieval and AACT tables for publication metadata.
3. **Cleaning Refinement:** Implemented regex word boundaries and complex negation rules to eliminate false positives in safety/efficacy detection.

**Next Steps:**  
The next phase will focus on filtering the 'Gold Standard' candidates for deep learning analysis using ICANN, prioritizing those with both high-confidence clinical evidence and complete structural data.